# 16.1 An input-output model

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

In simple maximum-profit production models such as the examples in [Chapter 1](../01/01.md), the
goods produced are distinct from the resources consumed, so that overall production is
limited in an obvious way by resources available. In a more realistic model of a complex
operation such as a steel mill or refinery, however, production is carried out at a series of
units; as a result, some of a production unit's inputs may be the outputs from other units.
For this situation we need a model that deals more generally with materials that may be
inputs or outputs, and with production activities that may involve several inputs and outputs
each.

We begin by developing an AMPL formulation in the usual row-wise (or constraintoriented)
way. Then we explain the columnwise (or variable-oriented) alternative, and
discuss refinements of the model.

**Formulation by constraints**

The definition of our model starts with a set of materials and a set of activities:

```ampl
set MAT;
set ACT;
```

The key data values are the input-output coefficients for all material-activity combinations
:

```ampl
param io {MAT,ACT};
```

If io[i,j] > 0, it is interpreted as the amount of material `i` produced (as an output) by
a unit of activity j. On the other hand, if io[i,j] < 0, it represents minus the amount
of material `i` consumed (as an input) by a unit of activity j. For example, a value of 10
represents 10 units of `i` produced per unit of j, while a value of -10 represents 10 units
consumed per unit of j. Of course, we can expect that for many combinations of `i` and `j`
we will have io[i,j] = 0, signifying that material `i` does not figure in activity `j` at all.

To see why we want to interpret io[i,j] in this manner, suppose we define
Run[j] to be the level at which we operate (run) activity j:

```ampl
param act_min {ACT} >= 0;
param act_max {j in ACT} >= act_min[j];
var Run {j in ACT} >= act_min[j], <= act_max[j];
```

Then io[i,j] * Run[j] is the total amount of material `i` produced (if io[i,j] > 0)
or minus the amount of material `i` consumed (if io[i,j] < 0) by activity j. Summing
over all activities, we see that

```ampl
set MAT;              # materials
set ACT;              # activities
param io {MAT,ACT};   # input-output coefficients
param revenue {ACT};
param act_min {ACT} >= 0;
param act_max {j in ACT} >= act_min[j];
var Run {j in ACT} >= act_min[j], <= act_max[j];
maximize Net_Profit: sum {j in ACT} revenue[j] * Run[j];
subject to Balance {i in MAT}:
       sum {j in ACT} io[i,j] * Run[j] = 0;
```

**[Figure 16-1](./16_1_an_inputoutput_model.ipynb#fig-16-1):** Input-output model by rows (iorow.mod).

```ampl
sum {j in ACT} io[i,j] * Run[j]
```

represents the amount of material `i` produced in the operation minus the amount consumed.
These amounts must balance, as expressed by the following constraint:

```ampl
subject to Balance {i in MAT}:
       sum {j in ACT} io[i,j] * Run[j] = 0;
```

What about the availability of resources, or the requirements for finished goods? These
are readily modeled through additional activities that represent the purchase or sale of
materials. A purchase activity for material `i` has no inputs and just `i` as an output; the
upper limit on Run[i] represents the amount of this resource available. Similarly, a sale
activity for material `i` has no outputs and just `i` as an input, and the lower limit on
Run[i] represents the amount of this good that must be produced for sale.

We complete the model by associating unit revenues with the activities. Sale activities
necessarily have positive revenues, while purchase and production activities have
negative revenues — that is, costs. The sum of unit revenues times activity levels gives
the total net profit of the operation:

```ampl
param revenue {ACT};
maximize Net_Profit: sum {j in ACT} revenue[j] * Run[j];
```

The completed model is shown in [Figure 16-1](./16_1_an_inputoutput_model.ipynb#fig-16-1).

**A columnwise formulation**

As our discussion of purchase and sale activities suggests, everything in this model
can be organized by activity. Specifically, for each activity `j` we have a decision variable
Run[j], a cost or income represented by revenue[j], limits act_min[j] and
act_max[j], and a collection of input-output coefficients io[i,j]. Changes such as
improving the yield of a unit, or acquiring a new source of supply, are accommodated by
adding an activity or by modifying the data for an activity.
In the formulation by rows, the activities' importance to this model is somewhat hidden.
While act_min[j] and act_max[j] appear in the declaration of the variables,
revenue[j] is in the objective, and the io[i,j] values are in the constraint declaration.
The columnwise alternative brings all of this information together, by adding obj
and `coeff` phrases to the `var` declaration:

```ampl
var Run {j in ACT} >= act_min[j], <= act_max[j],
       obj Net_Profit revenue[j],
       coeff {i in MAT} Balance[i] io[i,j];
```

The obj phrase says that in the objective function named Net_Profit, the variable
Run[j] has the coefficient revenue[j]; that is, the term revenue[j] * Run[j]
should be added in. The `coeff` phrase is a bit more complicated, because it is indexed
over a set. It says that for each material i, in the constraint Balance[i] the variable
Run[j] should have the coefficient io[i,j], so that the term io[i,j] * Run[j] is
added in. Together, these phrases describe all the coefficients of all the variables in the
linear program.

Since we have placed all the coefficients in the `var` declaration, we must remove
them from the other declarations:

```ampl
maximize Net_Profit;
subject to Balance {i in MAT}: to_come = 0;
```

The keyword `to_come` indicates where the terms io[i,j] * Run[j] generated by
the `var` declaration are to be "added in." You can think of `to_come` = 0 as a template
for the constraint, which will be filled out as the coefficients are declared. No template is
needed for the objective in this example, however, since it is exclusively the sum of the
terms revenue[j] * Run[j]. Templates may be written in a limited variety of ways,
as shown in [Section 16.3](./16_3_rules_for_columnwise_formulations.ipynb) below.

Because the obj and `coeff` phrases refer to Net_Profit and Balance, the `var`
declaration must come after the maximize and `subject to` declarations in the
columnwise formulation. The complete model is shown in [Figure 16-2](./16_1_an_inputoutput_model.ipynb#fig-16-2).

```ampl
set MAT;             # materials
set ACT;             # activities
param io {MAT,ACT};  # input-output coefficients
param revenue {ACT};
param act_min {ACT} >= 0;
param act_max {j in ACT} >= act_min[j];
maximize Net_Profit;
subject to Balance {i in MAT}: to_come = 0;
var Run {j in ACT} >= act_min[j], <= act_max[j],
       obj Net_Profit revenue[j],
       coeff {i in MAT} Balance[i] io[i,j];
```

**[Figure 16-2](./16_1_an_inputoutput_model.ipynb#fig-16-2):** Columnwise formulation (iocol1.mod).

**Refinements of the columnwise formulation**

The advantages of a columnwise approach become more evident as the model
becomes more complicated. As one example, consider what happens if we want to have
separate variables to represent sales of finished materials. We declare a subset of materials
that can be sold, and use it to index new collections of bounds, revenues and variables:

```ampl
set MATF within MAT;            # finished materials
param revenue {MATF} >= 0;
param sell_min {MATF} >= 0;
param sell_max {i in MATF} >= sell_min[i];
var Sell {i in MATF} >= sell_min[i], <= sell_max[i];
```

We may now dispense with the special sale activities previously described. Since the
remaining members of ACT represent purchase or production activities, we can introduce
a nonnegative parameter cost associated with them:

```ampl
param cost {ACT} >= 0;
```

In the row-wise approach, the new objective is written as

```ampl
maximize Net_Profit:
       sum {i in MATF} revenue[i] * Sell[i]
       - sum {j in ACT} cost[j] * Run[j];
```

to represent total sales revenue minus total raw material and production costs.

So far we seem to have improved upon the model in [Figure 16-1](./16_1_an_inputoutput_model.ipynb#fig-16-1). The composition of
net profit is more clearly modeled, and sales are restricted to explicitly designated finished
materials; also the optimal amounts sold are more easily examined apart from the
other variables, by a command such as display Sell. It remains to `fix` up the constraints.
We would like to say that the net output of material `i` from all activities, represented
as

```ampl
sum {j in ACT} io[i,j] * Run[j]
```

in [Figure 16-1](./16_1_an_inputoutput_model.ipynb#fig-16-1), must balance the amount sold — either Sell[i] if `i` is a finished material
, or zero. Thus the constraint declaration must be written:

```ampl
subject to Balance {i in MAT}:
       sum {j in ACT} io[i,j] * Run[j]
       = if i in MATF then Sell[i] else 0;
```

Unfortunately this constraint seems less clear than our original one, due to the complication
introduced by the if-then-else expression.

In the columnwise alternative, the objective and constraints are the same as in [Figure 16-2](./16_1_an_inputoutput_model.ipynb#fig-16-2),
while all the changes are reflected in the declarations of the variables:

```ampl
set MAT;             # materials
set ACT;             # activities
param io {MAT,ACT};  # input-output coefficients
set MATF within MAT; # finished materials
param revenue {MATF} >= 0;
param sell_min {MATF} >= 0;
param sell_max {i in MATF} >= sell_min[i];
param cost {ACT} >= 0;
param act_min {ACT} >= 0;
param act_max {j in ACT} >= act_min[j];
maximize Net_Profit;
subject to Balance {i in MAT}: to_come = 0;
var Run {j in ACT} >= act_min[j], <= act_max[j],
       obj Net_Profit -cost[j],
       coeff {i in MAT} Balance[i] io[i,j];
var Sell {i in MATF} >= sell_min[i], <= sell_max[i],
       obj Net_Profit revenue[i],
       coeff Balance[i] -1;
```

**[Figure 16-3](./16_1_an_inputoutput_model.ipynb#fig-16-3):** Columnwise formulation, with sales activities (iocol2.mod).

```ampl
var Run {j in ACT} >= act_min[j], <= act_max[j],
       obj Net_Profit -cost[j],
       coeff {i in MAT} Balance[i] io[i,j];

var Sell {i in MATF} >= sell_min[i], <= sell_max[i],
       obj Net_Profit revenue[i],
       coeff Balance[i] -1;
```

In this view, the variable Sell[i] represents the kind of sale activity that we previously
described, with only material `i` as input and no materials as output — hence the single
coefficient of -1 in constraint Balance[i]. We need not specify all the zero coefficients
for Sell[i]; a zero is assumed for any constraint not explicitly cited in a `coeff`
phrase in the declaration. The whole model is shown in [Figure 16-3](./16_1_an_inputoutput_model.ipynb#fig-16-3).

This example suggests that a columnwise approach is particularly suited to refinements
of the input-output model that distinguish different kinds of activities. It would be
easy to add another group of variables that represent purchases of raw materials, for
instance.

On the other hand, versions of the input-output model that involve numerous specialized
constraints would lend themselves more to a formulation by rows.